# Init

In [0]:
from pyspark.sql import functions as F

In [0]:
df = spark.table('projeto_combustiveis.bronze.mensal_brasil_desde_jan2013')
df.display()

# DICIONÁRIO DE MAPEAMENTO DE COLUNAS

In [0]:
dicionario_colunas = {
    "ms": "mes_referencia",
    "regio": "regiao",
    "municpio": "municipio",
    "nmero_de_postos_pesquisados": "numero_postos_pesquisados",
    "preo_mdio_revenda": "preco_medio_revenda",
    "desvio_padro_revenda": "desvio_padrao_revenda",
    "preo_mnimo_revenda": "preco_minimo_revenda",
    "preo_mximo_revenda": "preco_maximo_revenda",
    "margem_mdia_revenda": "margem_media_revenda",
    "coef_de_variao_revenda": "coef_variacao_revenda",
    "preo_mdio_distribuio": "preco_medio_distribuicao",
    "desvio_padro_distribuio": "desvio_padrao_distribuicao",
    "preo_mnimo_distribuio": "preco_minimo_distribuicao",
    "preo_mximo_distribuio": "preco_maximo_distribuicao",
    "coef_de_variao_distribuio": "coef_variacao_distribuicao"
}

# FUNÇÃO DE LIMPEZA E PADRONIZAÇÃO DA SILVER LAYER

In [0]:
def limpa_tabela_mensal(nome_tabela_bronze, nome_tabela_silver, dict_renomear):
    print(f"A iniciar o processamento da tabela: {nome_tabela_silver}...")
    
    # Leitura da tabela Bruta
    df = spark.read.table(f"projeto_combustiveis.bronze.{nome_tabela_bronze}")
    
    # Passo 1: Renomear colunas corrompidas na ingestão
    for antiga, nova in dict_renomear.items():
        if antiga in df.columns:
            df = df.withColumnRenamed(antiga, nova)
            
    # Passo 2: Aplicar TRIM
    colunas_texto = [coluna for coluna, tipo in df.dtypes if tipo == "string"]
    for col in colunas_texto:
        df = df.withColumn(col, F.trim(F.col(col)))
        
    # Passo 3: Converter a coluna de data
    if "mes_referencia" in df.columns:
        df = df.withColumn("mes_referencia", F.to_date(F.col("mes_referencia"), "yyyy-MM-dd"))
        
    # Passo 4: Tratar nulos ("-") e tipar a quantidade de postos (Inteiro)
    if "numero_postos_pesquisados" in df.columns:
        df = df.withColumn(
            "numero_postos_pesquisados", 
            F.when(F.col("numero_postos_pesquisados") == "-", None).otherwise(F.col("numero_postos_pesquisados")).cast("int")
        )
        
    # Passo 5: Tratar nulos ("-") e tipar dinamicamente as colunas financeiras (Decimal/Double)
    palavras_chave_numericas = ["preco", "desvio", "margem", "coef"]
    colunas_decimais = [c for c in df.columns if any(palavra in c for palavra in palavras_chave_numericas)]
    
    for col in colunas_decimais:
        df = df.withColumn(
            col,
            F.when(F.col(col) == "-", None).otherwise(F.col(col)).cast("double")
        )
        
    # Passo 6: Qualidade de Dados
    df = df.dropDuplicates()
    
    if "mes_referencia" in df.columns and "produto" in df.columns:
        df = df.dropna(subset=["mes_referencia", "produto"])
        
    # Passo 7: Governança
    df = df.withColumn("data_processamento_silver", F.current_timestamp())
    
    # Salvar na Camada silver
    df.write.mode("overwrite").saveAsTable(f"projeto_combustiveis.silver.{nome_tabela_silver}")
    print(f"Tabela {nome_tabela_silver} guardada com sucesso no Unity Catalog!\n")

# EXECUÇÃO DO PIPELINE PARA TABELAS HISTÓRICAS

In [0]:
limpa_tabela_mensal("mensal_brasil_desde_jan2013", "historico_mensal_brasil", dicionario_colunas)
limpa_tabela_mensal("mensal_regioes_desde_jan2013", "historico_mensal_regioes", dicionario_colunas)
limpa_tabela_mensal("mensal_estados_desde_jan2013", "historico_mensal_estados", dicionario_colunas)
limpa_tabela_mensal("mensal_municipios_desde_jan2022_2025", "historico_mensal_municipios_22_25", dicionario_colunas)
limpa_tabela_mensal("mensal_municipios_desde_jan2026", "historico_mensal_municipios_26", dicionario_colunas)

# UNION DAS TABELAS DE MUNICÍPIOS

In [0]:
df_mun_22_25 = spark.read.table("projeto_combustiveis.silver.historico_mensal_municipios_22_25")
df_mun_26 = spark.read.table("projeto_combustiveis.silver.historico_mensal_municipios_26")

df_municipios_completo = df_mun_22_25.unionByName(df_mun_26)

df_municipios_completo.write.mode("overwrite").saveAsTable("projeto_combustiveis.silver.historico_mensal_municipios")

# Remove as tabelas temporárias fragmentadas
spark.sql("DROP TABLE IF EXISTS projeto_combustiveis.silver.historico_mensal_municipios_22_25")
spark.sql("DROP TABLE IF EXISTS projeto_combustiveis.silver.historico_mensal_municipios_26")

# VERIFICAÇÃO EM SQL

In [0]:
%sql
select * from projeto_combustiveis.silver.historico_mensal_brasil limit(10)

In [0]:
%sql
select * from projeto_combustiveis.silver.historico_mensal_estados limit(10)

In [0]:
%sql
select * from projeto_combustiveis.silver.historico_mensal_municipios
order by mes_referencia asc
limit(10)

In [0]:
%sql
select * from projeto_combustiveis.silver.historico_mensal_regioes 
order by mes_referencia